# health_agent 项目技术背景说明文档

## 1. 项目概述

`health_agent` 是一个面向诊所、康复中心、理疗门店等场景的**本地部署版健康管理系统**。
从业务流程看，系统围绕以下闭环展开：

**客户建档 → 健康评估 → 到店/上门预约 → 仪器使用 → 满意度回访 → 查询导出/备份恢复**

从当前代码结构看，这个项目属于典型的：

* **单机部署应用**
* **轻量级 Web 管理系统**
* **后端与前端一体化项目**
* **SQLite 本地数据库驱动的业务系统**

它的核心特点是：**部署简单、依赖少、适合中小规模门店本地运行**。

## 2. 当前技术架构说明

## 2.1 总体架构

当前项目采用的是：

**Flask + SQLite + 原生 HTML/CSS/JavaScript + pandas/openpyxl**

可以把它理解成一个“轻量级单体应用”：

* 后端由 Flask 提供 HTTP 接口与静态文件服务
* 前端是单页式原生页面，放在 `static/` 目录中
* 数据库存储使用 SQLite，本地文件为 `medical_system.db`
* 报表导出依赖 pandas 与 openpyxl
* 本地启动体验通过 `launch.py + bat/pyw` 封装

从架构层面讲，它不是前后端分离的复杂微服务系统，而是**一个本地可执行、便于落地的轻量单体系统**。

## 2.2 后端技术背景

### Flask 作为 Web 服务框架

项目后端主程序是 `app.py`，使用 Flask 搭建接口与页面服务。
Flask 适合这种中小型业务系统，原因是：

* 上手快，适合快速开发业务原型
* 路由清晰，方便直接写 CRUD 接口
* 可以同时服务 API 和静态资源
* 与 Python 数据处理生态结合自然

Flask 官方文档说明，应用可以直接通过 `static` 目录提供静态资源；同时 Flask 的 `session` 机制本质上基于**签名 cookie**，依赖 `secret_key` 来保护会话不被篡改。当前项目正是按这种方式实现登录态的。 ([Flask 文档][1])

### 当前后端组织方式

当前后端仍然是**单文件主程序模式**：

* `app.py`：主入口、数据库初始化、校验函数、业务逻辑、路由
* `launch.py`：桌面启动器
* `requirements.txt`：依赖管理

这意味着当前项目的可维护性重点不在“多模块协作”，而在于：

* 逻辑集中
* 部署简单
* 修改路径短

适合本地工具型系统，但后续如果继续扩展，还是建议再做模块拆分。

---

## 2.3 前端技术背景

前端采用的是：

* `static/index.html`
* `static/app.js`

也就是**原生 HTML + CSS + JavaScript**，没有引入 Vue、React、Node.js、打包器等前端工程体系。

这套方案的好处是：

* 没有前端构建链路
* 本地部署非常简单
* 环境依赖少
* 对非纯技术运维人员更友好

它更像“后端驱动的轻量单页后台系统”，适合内部运营、表单录入、数据查询类场景。

---

## 2.4 数据库技术背景

### SQLite 作为嵌入式数据库

项目数据库是 SQLite，本地文件型数据库，默认库文件是 `medical_system.db`。

这类数据库非常适合：

* 单机部署
* 小团队使用
* 不需要复杂数据库运维
* 希望数据文件可直接备份、迁移、拷贝的场景

### 外键约束

SQLite 官方文档说明：**外键约束默认不是自动生效的**，应用需要在每个数据库连接上显式执行 `PRAGMA foreign_keys = ON`。当前项目已经在 `get_db()` 中这样做了。 ([sqlite.org][2])

### WAL 模式

当前项目也启用了：

```sql
PRAGMA journal_mode=WAL;
```

SQLite 官方说明，WAL（Write-Ahead Logging）模式的特点是：

* 写入先进入 WAL 日志文件
* 读写并发能力优于传统 rollback journal
* 引入了 checkpoint 机制
* WAL 文件是数据库持久状态的一部分，不能随便和主库文件分离复制 ([a1.sqlite.org][3])

这也是为什么你现在的项目在备份逻辑上改成了**SQLite backup API**，而不是简单复制 `.db` 文件。

### 在线备份 API

SQLite 官方文档指出，在线备份 API 相比直接复制数据库文件，对“运行中的数据库”更安全，能够避免简单文件拷贝在活跃数据库场景下带来的锁和损坏风险。当前代码中的 `create_db_backup()` 和 `restore_db_from_backup()` 已经体现出这种思路。 ([sqlite.org][4])

---

## 2.5 导出能力技术背景

当前系统导出 Excel 使用的是：

* `pandas`
* `openpyxl`

其中 pandas 官方文档说明，`DataFrame.to_excel()` 支持直接写入 `.xlsx` 文件；如果结合 `ExcelWriter`，可以把多个 DataFrame 输出到一个 Excel 的多个 sheet 中。当前项目的“查询导出”模块正是按这个模式组织的。 ([熊猫数据分析库][5])

---

## 2.6 当前安全与运维特征

从当前代码看，项目已经具备这些基础运维能力：

* 登录态校验（`session`）
* `secret_key`
* 审计日志表 `audit_logs`
* 数据库备份
* 数据库恢复
* 客户软删除（`is_deleted`）
* 应用日志 `logs/app.log`
* 启动日志 `logs/startup.log`
* 桌面启动封装 `launch.py`

这说明项目已经从“纯 demo”往“可实际使用的本地业务系统”迈进了一步。

不过从当前实现看，仍要注意一个事实：

**当前登录态是 session 化了，但密码仍然保存在 `system_settings` 表中，而不是独立用户表。**
所以从架构层面看，它现在属于“轻量单管理员本地系统”，还不是完整的 RBAC 多角色权限系统。

# 3. 代码层面的系统模块划分

从当前代码可以把系统分成 7 层理解：

## 3.1 展示层

* `static/index.html`
* `static/app.js`

负责：

* 页面展示
* 表单录入
* 调用 `/api/*`
* 列表分页、搜索、图表展示

## 3.2 接口层

* `app.py` 中所有 `/api/*`

负责：

* 接收前端请求
* 参数校验
* 访问数据库
* 返回 JSON

## 3.3 鉴权层

* `session`
* `@app.before_request`

负责：

* 检查登录状态
* 拦截未登录访问

## 3.4 业务规则层

体现在函数中，例如：

* `validate_customer_payload`
* `validate_appointment_payload`
* `is_project_home_allowed`
* `get_project_required_equipment_name`

## 3.5 数据层

* SQLite
* `init_db()`
* 各类 `CREATE TABLE`

## 3.6 数据输出层

* Excel 导出
* 备份恢复

## 3.7 运维支持层

* `logging`
* `audit_logs`
* `launch.py`

# 4. 数据库表清单（当前代码）

当前 `init_db()` 中实际涉及的表一共有 **17 张**：

1. `customers`
2. `equipment`
3. `appointments`
4. `equipment_usage`
5. `health_assessments`
6. `therapy_projects`
7. `service_projects`
8. `staff`
9. `home_appointments`
10. `db_backups`
11. `system_settings`
12. `audit_logs`
13. `project_rules`
14. `project_equipment_mapping`
15. `satisfaction_surveys`
16. `health_records`
17. `visit_checkins`

---

# 5. 各表主键与作用说明

| 表名                        | 主键          | 作用         |
| ------------------------- | ----------- | ---------- |
| customers                 | id          | 客户主档案      |
| equipment                 | id          | 设备主数据      |
| appointments              | id          | 到店预约记录     |
| equipment_usage           | id          | 仪器使用记录     |
| health_assessments        | id          | 健康评估档案     |
| therapy_projects          | id          | 理疗/治疗项目主数据 |
| service_projects          | id          | 服务项目补充表    |
| staff                     | id          | 服务人员主数据    |
| home_appointments         | id          | 上门预约记录     |
| db_backups                | id          | 数据库备份记录    |
| system_settings           | setting_key | 系统参数配置表    |
| audit_logs                | id          | 操作审计日志     |
| project_rules             | id          | 项目规则配置     |
| project_equipment_mapping | id          | 项目与设备映射配置  |
| satisfaction_surveys      | id          | 满意度回访记录    |
| health_records            | id          | 健康记录       |
| visit_checkins            | id          | 来访签到记录     |

---

# 6. 数据库中“真实外键关系”总览

这里要区分两层：

## 第一层：数据库里**真正声明了 FOREIGN KEY** 的关系

这部分是数据库级勾稽关系。

### 6.1 客户主表 customers 为核心父表

`customers.id` 被以下表引用：

* `appointments.customer_id -> customers.id`
* `equipment_usage.customer_id -> customers.id`
* `health_assessments.customer_id -> customers.id`
* `home_appointments.customer_id -> customers.id`
* `satisfaction_surveys.customer_id -> customers.id`
* `health_records.customer_id -> customers.id`
* `visit_checkins.customer_id -> customers.id`

也就是说：

**customers 是整个系统的“客户主索引表”**。

---

### 6.2 预约表 appointments 的关联

数据库中真正声明的外键有：

* `appointments.customer_id -> customers.id`
* `appointments.equipment_id -> equipment.id`

说明：

* 一个客户可以有多条预约
* 一台设备可以被多条预约引用（不同时间段）

---

### 6.3 设备使用表 equipment_usage 的关联

数据库中真正声明的外键有：

* `equipment_usage.customer_id -> customers.id`
* `equipment_usage.equipment_id -> equipment.id`
* `equipment_usage.appointment_id -> appointments.id`

说明：

* 设备使用记录可追溯到客户
* 可追溯到具体设备
* 也可以追溯到来源预约

---

### 6.4 健康评估表 health_assessments

* `health_assessments.customer_id -> customers.id`

说明：

* 一个客户可以有多次健康评估
* 这是典型的一对多关系

---

### 6.5 上门预约表 home_appointments

数据库中声明了：

* `home_appointments.customer_id -> customers.id`
* `home_appointments.project_id -> therapy_projects.id`
* `home_appointments.staff_id -> staff.id`

说明：

* 上门预约同时挂接客户、项目、服务人员三类主数据

---

### 6.6 满意度表 satisfaction_surveys

数据库中声明了：

* `satisfaction_surveys.customer_id -> customers.id`
* `satisfaction_surveys.appointment_id -> appointments.id`

说明：

* 满意度可以关联客户
* 也可以回溯到一次具体的到店预约

---

### 6.7 健康记录表 health_records

* `health_records.customer_id -> customers.id`

### 6.8 来访签到表 visit_checkins

* `visit_checkins.customer_id -> customers.id`

---

# 7. 代码层“逻辑关联”与数据库外键的区别

这是你汇报里最值得强调的一点。

## 有些字段在代码中被当成关联键使用，但数据库层**没有真正声明外键**

例如：

### 7.1 appointments 表

虽然有字段：

* `project_id`
* `staff_id`

但当前建表语句里**没有显式声明**

* `project_id -> therapy_projects.id`
* `staff_id -> staff.id`

也就是说：

* **代码逻辑上是关联的**
* **数据库约束上不是强制的**

---

### 7.2 equipment_usage 表

虽然有字段：

* `project_id`
* `staff_id`

但当前建表里也没有显式外键到：

* `therapy_projects.id`
* `staff.id`

因此它们是：

* 业务层面的关联字段
* 不是数据库强约束字段

---

### 7.3 project_rules 与 project_equipment_mapping

这两张表不是通过 `project_id`、`equipment_id` 做关联，而是通过：

* `project_name`
* `equipment_name`

进行**文本级业务映射**

这说明当前系统在“项目规则配置”层面采用的是：

* **轻量配置驱动**
* 而不是严格的关系型主键映射

这对快速落地有好处，但从规范化角度看，后续可以再演进为 `project_id / equipment_id` 关联。

---

# 8. 表勾稽关系（可直接用于 PPT）

## 8.1 以客户为中心的业务勾稽

**customers（客户主档）**
是整个系统的业务中心，向外发散到：

* `health_assessments`：健康评估
* `health_records`：健康记录
* `appointments`：到店预约
* `home_appointments`：上门预约
* `equipment_usage`：设备使用
* `visit_checkins`：签到
* `satisfaction_surveys`：满意度回访

可以概括为：

**1 个客户，对应多条业务记录。**

---

## 8.2 预约驱动的业务勾稽

`appointments` 是门店业务执行的核心表，向外连接：

* `customers`
* `equipment`
* （逻辑上）`therapy_projects`
* （逻辑上）`staff`
* `equipment_usage`
* `satisfaction_surveys`

可以理解为：

**预约表是“资源调度中心”**

它连接了客户、设备、项目、人员，并能衍生出后续使用记录和满意度回访。

---

## 8.3 项目与资源配置勾稽

项目相关表包括：

* `therapy_projects`：项目主数据
* `project_rules`：项目规则（是否允许上门）
* `project_equipment_mapping`：项目与设备映射
* `service_projects`：服务项目补充表

这几张表共同构成：

**项目配置层**

它们决定了：

* 某项目是否能上门
* 某项目需要什么设备
* 前端展示时哪些项目可选

---

## 8.4 运维支持勾稽

运维相关表包括：

* `system_settings`：系统参数
* `db_backups`：备份记录
* `audit_logs`：审计日志

这三张表不直接参与客户业务流程，但支撑：

* 配置管理
* 数据安全
* 操作追踪

# 9. 推荐你在汇报里使用的关系图口径

你可以直接这么讲：

```text
客户主表 customers 是整个系统的数据中心。
围绕客户，系统沉淀健康评估、健康记录、预约、上门预约、签到、设备使用、满意度等多类业务数据。

其中 appointments 是门店履约中心，连接客户、设备、项目和人员；
home_appointments 是上门服务履约中心，连接客户、项目和服务人员；
project_rules 与 project_equipment_mapping 是项目配置层；
system_settings、db_backups、audit_logs 构成系统运维保障层。
```

---

# 10. 可以直接放进汇报的总结页

## 技术总结

当前项目采用 **Flask + SQLite + 原生前端** 的轻量单体架构，特点是：

* 部署简单
* 本地运行成本低
* 功能闭环完整
* 适合门店级健康管理场景快速落地

## 数据模型总结

数据库以 `customers` 为核心主表，形成：

* 客户主数据层
* 健康档案层
* 预约与履约层
* 项目配置层
* 运维保障层

## 设计特点总结

当前系统同时具备两类关联方式：

1. **数据库强约束关联**：通过外键维护主从关系
2. **代码逻辑关联**：通过业务字段和配置表实现灵活控制

这使得系统既保留了基本数据一致性，又具备较强的本地化配置灵活性。

---

如果你要，我下一步可以直接继续帮你整理成一版更适合 PPT 的内容，例如：

**“一页技术架构图 + 一页数据库 ER 关系图 + 一页表关系说明表”**。

[1]: https://flask.palletsprojects.com/en/stable/quickstart/?utm_source=chatgpt.com "Quickstart — Flask Documentation (3.1.x)"
[2]: https://sqlite.org/foreignkeys.html?utm_source=chatgpt.com "SQLite Foreign Key Support"
[3]: https://a1.sqlite.org/wal.html?utm_source=chatgpt.com "Write-Ahead Logging"
[4]: https://www.sqlite.org/backup.html?utm_source=chatgpt.com "SQLite Backup API"
[5]: https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.to_excel.html?utm_source=chatgpt.com "pandas.DataFrame.to_excel — pandas 3.0.1 documentation"

# 健康管理系统技术架构图

```mermaid
flowchart TB
    A[用户/运营人员<br/>Browser 浏览器] --> B[前端展示层<br/>static/index.html<br/>static/app.js]

    B --> C[Flask 应用层<br/>app.py]
    C --> C1[登录鉴权<br/>session / before_request]
    C --> C2[业务接口层<br/>客户/健康档案/预约/上门/使用/满意度]
    C --> C3[校验与规则层<br/>validate_* / 项目规则 / 冲突校验]
    C --> C4[数据导出与系统管理<br/>Excel 导出 / 备份 / 恢复 / 审计日志]

    C --> D[SQLite 数据层<br/>medical_system.db]
    C --> E[文件输出层<br/>exports/ 导出文件]
    C --> F[日志与运维层<br/>logs/app.log<br/>audit_logs<br/>db_backups]

    G[桌面启动器<br/>launch.py / 启动医疗系统.bat / pyw] --> C
    G --> A
```

### 技术架构说明

本项目采用 **Flask + SQLite + 原生 HTML/JS** 的轻量单体架构，适合本地单机部署。
前端负责页面展示与表单交互，后端通过 Flask 提供 API、业务校验、导出与备份能力，底层使用 SQLite 持久化业务数据。

### 核心特点

* **部署简单**：无需 Node.js，无需前端构建流程
* **本地可落地**：适合诊所、康复中心、理疗门店本地使用
* **业务闭环完整**：支持建档、评估、预约、履约、回访、导出、备份恢复
* **运维轻量**：通过启动器封装桌面运行体验，降低使用门槛

## 备注
“系统不是复杂的云原生架构，而是一个轻量、可本地快速落地的业务系统”。
可以突出三层：

1. 展示层：HTML/JS
2. 应用层：Flask
3. 数据层：SQLite + 导出/日志/备份

# 数据库 ER 关系图（核心业务表）**

```mermaid
erDiagram
    CUSTOMERS ||--o{ APPOINTMENTS : "customer_id"
    CUSTOMERS ||--o{ HOME_APPOINTMENTS : "customer_id"
    CUSTOMERS ||--o{ HEALTH_ASSESSMENTS : "customer_id"
    CUSTOMERS ||--o{ HEALTH_RECORDS : "customer_id"
    CUSTOMERS ||--o{ VISIT_CHECKINS : "customer_id"
    CUSTOMERS ||--o{ EQUIPMENT_USAGE : "customer_id"
    CUSTOMERS ||--o{ SATISFACTION_SURVEYS : "customer_id"

    EQUIPMENT ||--o{ APPOINTMENTS : "equipment_id"
    EQUIPMENT ||--o{ EQUIPMENT_USAGE : "equipment_id"

    APPOINTMENTS ||--o{ EQUIPMENT_USAGE : "appointment_id"
    APPOINTMENTS ||--o{ SATISFACTION_SURVEYS : "appointment_id"

    THERAPY_PROJECTS ||--o{ HOME_APPOINTMENTS : "project_id"
    STAFF ||--o{ HOME_APPOINTMENTS : "staff_id"

    CUSTOMERS {
        int id PK
        string name
        string id_card
        string phone
    }

    APPOINTMENTS {
        int id PK
        int customer_id FK
        int equipment_id FK
        int project_id
        int staff_id
    }

    HOME_APPOINTMENTS {
        int id PK
        int customer_id FK
        int project_id FK
        int staff_id FK
    }

    HEALTH_ASSESSMENTS {
        int id PK
        int customer_id FK
    }

    HEALTH_RECORDS {
        int id PK
        int customer_id FK
    }

    VISIT_CHECKINS {
        int id PK
        int customer_id FK
    }

    EQUIPMENT {
        int id PK
        string name
    }

    EQUIPMENT_USAGE {
        int id PK
        int customer_id FK
        int equipment_id FK
        int appointment_id FK
        int project_id
        int staff_id
    }

    SATISFACTION_SURVEYS {
        int id PK
        int customer_id FK
        int appointment_id FK
    }

    THERAPY_PROJECTS {
        int id PK
        string name
    }

    STAFF {
        int id PK
        string name
    }
```

### 核心数据中心

`customers` 是整个系统的**数据中心表**，向外关联健康评估、健康记录、预约、上门预约、签到、设备使用、满意度等业务表。

### 履约中心

* `appointments` 是**到店预约中心**
* `home_appointments` 是**上门预约中心**

### 履约追踪

* `equipment_usage` 可追溯到客户、设备及具体预约
* `satisfaction_surveys` 可追溯到客户及预约

### 说明

图中展示的是**核心业务表 ER 关系**。
配置类表（如 `project_rules`、`project_equipment_mapping`、`system_settings`、`db_backups`、`audit_logs`）没有全部放入 ER 图，以免页面过于复杂，适合在下一页用说明表解释。

## 备注

这一页重点讲两个中心：

* **客户中心**：以 `customers` 为主
* **预约履约中心**：以 `appointments / home_appointments` 为主

同时要强调：

* 有些关联是数据库强外键
* 有些是代码逻辑关联（比如 `project_rules`、`project_equipment_mapping`）

# 数据库表关系说明

## 建议表格内容

| 表名                        | 主键          | 表定位      | 关联对象                                                                                                                 | 关联方式                                                                               | 说明         |
| ------------------------- | ----------- | -------- | -------------------------------------------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------- | ---------- |
| customers                 | id          | 客户主档表    | appointments、home_appointments、health_assessments、health_records、visit_checkins、equipment_usage、satisfaction_surveys | 主键 `id` 被各业务表外键引用                                                                  | 全系统数据中心    |
| appointments              | id          | 到店预约表    | customers、equipment、equipment_usage、satisfaction_surveys                                                             | `customer_id`、`equipment_id` 为数据库外键；`project_id`、`staff_id` 为代码逻辑关联                | 门店履约中心     |
| home_appointments         | id          | 上门预约表    | customers、therapy_projects、staff                                                                                     | `customer_id`、`project_id`、`staff_id` 为数据库外键                                       | 上门履约中心     |
| health_assessments        | id          | 健康评估表    | customers                                                                                                            | `customer_id -> customers.id`                                                      | 一位客户可有多次评估 |
| health_records            | id          | 健康记录表    | customers                                                                                                            | `customer_id -> customers.id`                                                      | 用于健康指标沉淀   |
| visit_checkins            | id          | 来访签到表    | customers                                                                                                            | `customer_id -> customers.id`                                                      | 记录到店签到行为   |
| equipment                 | id          | 设备主数据表   | appointments、equipment_usage                                                                                         | `equipment_id` 被预约和使用记录引用                                                          | 设备资源主表     |
| equipment_usage           | id          | 设备使用记录表  | customers、equipment、appointments                                                                                     | `customer_id`、`equipment_id`、`appointment_id` 为数据库外键；`project_id`、`staff_id` 为逻辑关联 | 履约执行记录     |
| satisfaction_surveys      | id          | 满意度回访表   | customers、appointments                                                                                               | `customer_id`、`appointment_id` 为数据库外键                                              | 服务闭环追踪     |
| therapy_projects          | id          | 理疗/服务项目表 | home_appointments                                                                                                    | `project_id -> therapy_projects.id`；部分场景被代码逻辑引用                                    | 项目主数据      |
| service_projects          | id          | 服务项目补充表  | 前端项目展示                                                                                                               | 代码层合并展示                                                                            | 并非核心事务表    |
| staff                     | id          | 服务人员表    | home_appointments                                                                                                    | `staff_id -> staff.id`；部分场景在代码逻辑中参与调度                                              | 人员主数据      |
| project_rules             | id          | 项目规则配置表  | therapy_projects                                                                                                     | 按 `project_name` 文本逻辑关联                                                            | 定义项目是否允许上门 |
| project_equipment_mapping | id          | 项目-设备映射表 | therapy_projects、equipment                                                                                           | 按 `project_name`、`equipment_name` 文本逻辑关联                                           | 定义项目需要的设备  |
| system_settings           | setting_key | 系统配置表    | 登录、备份路径等系统功能                                                                                                         | 配置读取                                                                               | 系统参数中心     |
| db_backups                | id          | 备份记录表    | 系统备份恢复模块                                                                                                             | 备份文件记录                                                                             | 运维保障表      |
| audit_logs                | id          | 审计日志表    | 登录、取消预约、导出、备份等操作                                                                                                     | 业务操作写入                                                                             | 安全审计表      |

## 页面总结文案

### 关系特点总结

数据库关系可分为两类：

**1）数据库强外键关系**
用于保证核心业务数据一致性，例如客户、预约、设备、满意度、签到等主流程表。

**2）代码逻辑关联关系**
用于支持更灵活的配置驱动，例如：

* `project_rules`
* `project_equipment_mapping`

### 总体评价

当前数据模型已经形成：

* **客户主数据层**
* **预约履约层**
* **健康档案层**
* **配置规则层**
* **运维保障层**

这套设计既保证了核心流程的勾稽完整性，也兼顾了本地部署项目需要的灵活性。

---

# 备注

本系统采用轻量单体架构，技术上以 Flask 作为业务服务层、SQLite 作为本地数据库、原生 HTML/JS 作为前端展示层，适合门店级健康管理场景快速部署。数据库设计上，以 `customers` 作为主数据中心，向外延伸健康评估、预约、上门服务、设备使用、满意度和签到等业务表；同时通过 `project_rules` 和 `project_equipment_mapping` 实现项目规则和资源配置管理，最终形成“建档—评估—预约—履约—回访—导出/备份”的完整闭环。